# GPU V3 — Tăng tốc Non-Local Means bằng Cross-Correlation

Notebook này đánh giá **GPU V3** của thuật toán Non-Local Means (NLM).

## Mục tiêu của V3

Ba phiên bản GPU được xây dựng theo ba hướng tối ưu khác nhau:

- **GPU V1 — Parallelism:** mỗi CUDA thread xử lý một output pixel.
- **GPU V2 — Memory reuse:** dùng Shared Memory với tile + halo để giảm truy cập lặp vào Global Memory.
- **GPU V3 — Computation reformulation:** biến đổi công thức patch distance để tận dụng **patch energy + cross-correlation**.

V3 sử dụng đẳng thức:

\[
\|P-Q\|^2
=
\|P\|^2
+
\|Q\|^2
-
2\langle P,Q\rangle
\]

Trong đó:

- \(\|P\|^2\): năng lượng của reference patch;
- \(\|Q\|^2\): năng lượng của candidate patch;
- \(\langle P,Q\rangle\): cross-correlation / dot product giữa hai patch.

Ý tưởng này được lấy cảm hứng từ repository tham khảo **Fast Non-Local Means and Asymptotic Non-Local Means**. Implementation trong project này **không copy nguyên code MATLAB**, mà giữ cùng semantics của GPU V1/V2 để có thể kiểm tra correctness trực tiếp.

## Cấu hình benchmark chính

- Image: `512 × 512`
- Patch: `7 × 7`
- Search window: `21 × 21`
- CUDA block: `16 × 16`
- V3 displacement batch mặc định: `32`

## 1. Chuẩn bị môi trường

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Thư mục gốc dự án:", PROJECT_ROOT)

### Import thư viện và các implementation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cupy as cp

from src.nlm.config import NLMConfig

from src.nlm.image_utils import (
    add_gaussian_noise,
    create_camera_test_image,
)

from src.nlm.metrics import (
    compare_outputs,
    evaluate_denoising,
)

from src.nlm.benchmark import (
    benchmark_cuda_kernel,
    benchmark_gpu_end_to_end,
)

from src.nlm.gpu_v1 import (
    nlm_gpu_v1,
    create_gpu_v1_kernel_launcher,
)

from src.nlm.gpu_v2 import (
    nlm_gpu_v2,
    prepare_gpu_v2_kernel_launch,
)

from src.nlm.gpu_v3 import (
    nlm_gpu_v3,
    prepare_gpu_v3_pipeline_launch,
    create_gpu_v3_kernel_launcher,
)

print("Import thành công.")

### Kiểm tra GPU CUDA

In [ ]:
device = cp.cuda.Device()
properties = cp.cuda.runtime.getDeviceProperties(device.id)

device_name = properties["name"]
if isinstance(device_name, bytes):
    device_name = device_name.decode()

print("CUDA device ID:", device.id)
print("Tên GPU:", device_name)
print(
    "Compute capability:",
    f'{properties["major"]}.{properties["minor"]}',
)

## 2. Cấu hình thí nghiệm

Ở V3, ta dùng workload lớn hơn cấu hình `3×3 / 7×7` ban đầu.

Với:

- patch `7×7` → 49 phần tử mỗi patch;
- search window `21×21` → 441 candidate positions;

nếu tính patch distance trực tiếp, mỗi output pixel có thể cần tới:

\[
441 \times 49 = 21{,}609
\]

phép so sánh phần tử patch.

Cấu hình lớn này giúp làm rõ bottleneck của patch-distance computation và tạo điều kiện để đánh giá V3.

In [ ]:
config = NLMConfig(
    image_size=512,
    noise_sigma=0.08,
    patch_size=7,
    search_window_size=21,
    h=0.12,
    random_seed=42,
    block_size_x=16,
    block_size_y=16,
)

V3_BATCH_SIZE = 32

patch_radius = config.patch_radius
search_radius = config.search_radius
padding_radius = config.padding_radius

print("Kích thước ảnh:", config.image_size)
print("Patch size:", config.patch_size)
print("Search window:", config.search_window_size)
print("Patch radius:", patch_radius)
print("Search radius:", search_radius)
print("Padding / halo radius:", padding_radius)
print("CUDA block:", config.block_size)
print("V3 displacement batch size:", V3_BATCH_SIZE)

### So sánh khối lượng tính toán với cấu hình nhỏ

In [ ]:
small_patch_size = 3
small_search_window = 7

small_work_per_pixel = (
    small_patch_size ** 2
    * small_search_window ** 2
)

large_work_per_pixel = (
    config.patch_size ** 2
    * config.search_window_size ** 2
)

workload_ratio = (
    large_work_per_pixel
    / small_work_per_pixel
)

print(
    "Cấu hình 3x3 / 7x7:",
    small_work_per_pixel,
    "patch-element comparisons / output pixel",
)

print(
    "Cấu hình 7x7 / 21x21:",
    large_work_per_pixel,
    "patch-element comparisons / output pixel",
)

print(
    "Khối lượng tính trực tiếp tăng khoảng:",
    f"{workload_ratio:.1f}×",
)

## 3. Tạo ảnh đầu vào dùng chung

GPU V1, V2 và V3 phải dùng **cùng một ảnh nhiễu** để việc so sánh là công bằng.

`seed` được cố định để mỗi lần chạy notebook đều tạo cùng dữ liệu đầu vào.

In [ ]:
clean_image = create_camera_test_image(
    image_size=config.image_size,
)

noisy_image = add_gaussian_noise(
    image=clean_image,
    sigma=config.noise_sigma,
    seed=config.random_seed,
)

clean_image = np.ascontiguousarray(
    clean_image,
    dtype=np.float32,
)

noisy_image = np.ascontiguousarray(
    noisy_image,
    dtype=np.float32,
)

print("Ảnh sạch:")
print("  shape:", clean_image.shape)
print("  dtype:", clean_image.dtype)
print(
    "  range:",
    float(clean_image.min()),
    "→",
    float(clean_image.max()),
)

print("\nẢnh nhiễu:")
print("  shape:", noisy_image.shape)
print("  dtype:", noisy_image.dtype)
print(
    "  range:",
    float(noisy_image.min()),
    "→",
    float(noisy_image.max()),
)

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(
    noisy_image,
    cmap="gray",
    vmin=0.0,
    vmax=1.0,
)
plt.title("Ảnh nhiễu đầu vào")
plt.axis("off")
plt.show()

## 4. Ý tưởng GPU V3 — Cross-Correlation

Ở GPU V1/V2, patch distance về bản chất vẫn được tính trực tiếp:

\[
D(P,Q)
=
\frac{1}{|P|}
\sum_{k \in P}
(P_k-Q_k)^2
\]

GPU V3 viết lại thành:

\[
D(P,Q)
=
\frac{
\|P\|^2
+
\|Q\|^2
-
2\langle P,Q\rangle
}{|P|}
\]

### Pipeline của V3

```text
Padded image
    │
    ├──> Patch-energy map: ||P||²
    │
    └──> Batch nhiều search displacements
             │
             ├──> Product maps: I(x) × I(x+d)
             │
             ├──> Horizontal box sum
             │
             ├──> Vertical accumulation
             │       → cross-correlation <P,Q>
             │
             ├──> Patch distance
             │       ||P||² + ||Q||² - 2<P,Q>
             │
             └──> NLM weight + weighted accumulation
    │
    └──> Normalize output
```

Điểm quan trọng so với V3 running-sum trước đó là **nhiều displacement được xử lý theo batch** thay vì tạo một nhóm kernel riêng cho từng displacement.

### Kiểm tra cấu trúc batch của V3

In [ ]:
v3_prepared = prepare_gpu_v3_pipeline_launch(
    image=noisy_image,
    patch_size=config.patch_size,
    search_window_size=config.search_window_size,
    h=config.h,
    block_size=config.block_size,
    displacement_batch_size=V3_BATCH_SIZE,
)

number_of_displacements = (
    v3_prepared["number_of_displacements"]
)

number_of_batches = (
    v3_prepared["number_of_batches"]
)

effective_batch_size = (
    v3_prepared["displacement_batch_size"]
)

estimated_pipeline_launches = (
    1                      # patch-energy kernel
    + 3 * number_of_batches
    + 1                    # normalize kernel
)

print(
    "Số search displacements:",
    number_of_displacements,
)

print(
    "Batch size:",
    effective_batch_size,
)

print(
    "Số batch:",
    number_of_batches,
)

print(
    "Số CUDA kernel launches ước tính cho pipeline:",
    estimated_pipeline_launches,
)

Với search window `21×21`, có **441 displacement**.

Nếu `batch_size = 32`, 441 displacement được chia thành khoảng **14 batch**.  
Mỗi batch dùng một nhóm kernel để xử lý nhiều displacement cùng lúc.

Đây là điểm cần benchmark: batching giảm số lần launch kernel, nhưng batch lớn hơn cũng làm tăng kích thước intermediate buffers và memory traffic.

## 5. Chạy GPU V1, GPU V2 và GPU V3

In [ ]:
denoised_gpu_v1 = nlm_gpu_v1(
    image=noisy_image,
    patch_size=config.patch_size,
    search_window_size=config.search_window_size,
    h=config.h,
    block_size=config.block_size,
)

denoised_gpu_v2 = nlm_gpu_v2(
    image=noisy_image,
    patch_size=config.patch_size,
    search_window_size=config.search_window_size,
    h=config.h,
    block_size=config.block_size,
)

denoised_gpu_v3 = nlm_gpu_v3(
    image=noisy_image,
    patch_size=config.patch_size,
    search_window_size=config.search_window_size,
    h=config.h,
    block_size=config.block_size,
    displacement_batch_size=V3_BATCH_SIZE,
)

print("GPU V1:", denoised_gpu_v1.shape, denoised_gpu_v1.dtype)
print("GPU V2:", denoised_gpu_v2.shape, denoised_gpu_v2.dtype)
print("GPU V3:", denoised_gpu_v3.shape, denoised_gpu_v3.dtype)

### So sánh trực quan

In [ ]:
fig, axes = plt.subplots(
    1,
    4,
    figsize=(16, 4),
)

images = [
    noisy_image,
    denoised_gpu_v1,
    denoised_gpu_v2,
    denoised_gpu_v3,
]

titles = [
    "Ảnh nhiễu",
    "GPU V1",
    "GPU V2",
    "GPU V3",
]

for ax, image, title in zip(
    axes,
    images,
    titles,
):
    ax.imshow(
        image,
        cmap="gray",
        vmin=0.0,
        vmax=1.0,
    )
    ax.set_title(title)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 6. Kiểm tra correctness

Mục tiêu của V3 là thay đổi **cách tính**, không thay đổi kết quả NLM.

Vì vậy V3 phải khớp với GPU V1/V2 trong sai số floating-point cho phép.

In [ ]:
gpu_v3_vs_v1 = compare_outputs(
    reference=denoised_gpu_v1,
    candidate=denoised_gpu_v3,
    atol=5e-5,
    rtol=5e-5,
)

gpu_v3_vs_v2 = compare_outputs(
    reference=denoised_gpu_v2,
    candidate=denoised_gpu_v3,
    atol=5e-5,
    rtol=5e-5,
)

print("GPU V3 so với GPU V1:")
display(gpu_v3_vs_v1)

print("GPU V3 so với GPU V2:")
display(gpu_v3_vs_v2)

In [ ]:
max_abs_error_v3_v1 = float(
    np.max(
        np.abs(
            denoised_gpu_v3
            - denoised_gpu_v1
        )
    )
)

max_abs_error_v3_v2 = float(
    np.max(
        np.abs(
            denoised_gpu_v3
            - denoised_gpu_v2
        )
    )
)

correctness_table = pd.DataFrame(
    [
        {
            "So sánh": "GPU V3 vs GPU V1",
            "Max absolute error": (
                max_abs_error_v3_v1
            ),
            "Allclose": np.allclose(
                denoised_gpu_v3,
                denoised_gpu_v1,
                atol=5e-5,
                rtol=5e-5,
            ),
        },
        {
            "So sánh": "GPU V3 vs GPU V2",
            "Max absolute error": (
                max_abs_error_v3_v2
            ),
            "Allclose": np.allclose(
                denoised_gpu_v3,
                denoised_gpu_v2,
                atol=5e-5,
                rtol=5e-5,
            ),
        },
    ]
)

correctness_table

In [ ]:
error_map = np.abs(
    denoised_gpu_v3
    - denoised_gpu_v2
)

plt.figure(figsize=(6, 6))
plt.imshow(error_map)
plt.title("Absolute error map — GPU V3 so với GPU V2")
plt.axis("off")
plt.colorbar(label="Absolute error")
plt.show()

## 7. Đánh giá chất lượng denoising

PSNR/SSIM được dùng để kiểm tra rằng việc reformulate patch distance không làm thay đổi đáng kể chất lượng ảnh.

Ở đây mục tiêu **không phải** chứng minh V3 cho ảnh đẹp hơn V1/V2, mà là xác nhận ba implementation đang thực hiện cùng một phép lọc NLM.

In [ ]:
gpu_v1_quality = evaluate_denoising(
    clean=clean_image,
    noisy=noisy_image,
    denoised=denoised_gpu_v1,
)

gpu_v2_quality = evaluate_denoising(
    clean=clean_image,
    noisy=noisy_image,
    denoised=denoised_gpu_v2,
)

gpu_v3_quality = evaluate_denoising(
    clean=clean_image,
    noisy=noisy_image,
    denoised=denoised_gpu_v3,
)

quality_table = pd.DataFrame(
    [
        {
            "Implementation": "GPU V1",
            **gpu_v1_quality,
        },
        {
            "Implementation": "GPU V2",
            **gpu_v2_quality,
        },
        {
            "Implementation": "GPU V3",
            **gpu_v3_quality,
        },
    ]
)

quality_table

## 8. Benchmark GPU compute-only

### Phạm vi đo

`benchmark_cuda_kernel()` dùng CUDA Events.

- GPU V1: đo CUDA kernel của V1.
- GPU V2: đo CUDA kernel của V2.
- GPU V3: đo **toàn bộ multi-kernel compute pipeline** của V3.

Các input/buffer GPU được chuẩn bị trước nên phạm vi này không bao gồm:

- CPU padding;
- Host-to-Device transfer;
- Device-to-Host transfer;
- one-time allocation ở phía Python.

Vì V3 có nhiều kernel, notebook dùng tên **GPU compute-only**, không gọi đây là “single-kernel runtime”.

In [ ]:
v1_launcher, v1_output_gpu = (
    create_gpu_v1_kernel_launcher(
        image=noisy_image,
        patch_size=config.patch_size,
        search_window_size=config.search_window_size,
        h=config.h,
        block_size=config.block_size,
    )
)

v2_prepared = prepare_gpu_v2_kernel_launch(
    image=noisy_image,
    patch_size=config.patch_size,
    search_window_size=config.search_window_size,
    h=config.h,
    block_size=config.block_size,
)

v2_launcher = v2_prepared[
    "kernel_launcher"
]

v3_prepared = prepare_gpu_v3_pipeline_launch(
    image=noisy_image,
    patch_size=config.patch_size,
    search_window_size=config.search_window_size,
    h=config.h,
    block_size=config.block_size,
    displacement_batch_size=V3_BATCH_SIZE,
)

v3_launcher = v3_prepared[
    "kernel_launcher"
]

print("Đã chuẩn bị xong V1/V2/V3 GPU launchers.")

### Pre-warm trước khi đo

In [ ]:
PREWARM_RUNS = 15

for _ in range(PREWARM_RUNS):
    v1_launcher()

for _ in range(PREWARM_RUNS):
    v2_launcher()

for _ in range(PREWARM_RUNS):
    v3_launcher()

cp.cuda.get_current_stream().synchronize()

print(
    f"Đã pre-warm mỗi implementation {PREWARM_RUNS} lần."
)

In [ ]:
v1_compute_benchmark = benchmark_cuda_kernel(
    kernel_launcher=v1_launcher,
    warmup_runs=10,
    measured_runs=50,
)

v2_compute_benchmark = benchmark_cuda_kernel(
    kernel_launcher=v2_launcher,
    warmup_runs=10,
    measured_runs=50,
)

v3_compute_benchmark = benchmark_cuda_kernel(
    kernel_launcher=v3_launcher,
    warmup_runs=10,
    measured_runs=50,
)

print("GPU V1 compute-only:")
display(v1_compute_benchmark)

print("GPU V2 compute-only:")
display(v2_compute_benchmark)

print("GPU V3 compute-only:")
display(v3_compute_benchmark)

In [ ]:
v1_compute_ms = (
    v1_compute_benchmark["median_ms"]
)

v2_compute_ms = (
    v2_compute_benchmark["median_ms"]
)

v3_compute_ms = (
    v3_compute_benchmark["median_ms"]
)

compute_table = pd.DataFrame(
    [
        {
            "Implementation": "GPU V1",
            "Tối ưu chính": (
                "Pixel-level parallelism"
            ),
            "Median compute-only (ms)": (
                v1_compute_ms
            ),
            "Speedup so với V1": 1.0,
        },
        {
            "Implementation": "GPU V2",
            "Tối ưu chính": (
                "Shared Memory reuse"
            ),
            "Median compute-only (ms)": (
                v2_compute_ms
            ),
            "Speedup so với V1": (
                v1_compute_ms
                / v2_compute_ms
            ),
        },
        {
            "Implementation": "GPU V3",
            "Tối ưu chính": (
                "Cross-correlation reformulation"
            ),
            "Median compute-only (ms)": (
                v3_compute_ms
            ),
            "Speedup so với V1": (
                v1_compute_ms
                / v3_compute_ms
            ),
        },
    ]
)

compute_table

In [ ]:
labels = ["GPU V1", "GPU V2", "GPU V3"]
compute_values = [
    v1_compute_ms,
    v2_compute_ms,
    v3_compute_ms,
]

plt.figure(figsize=(8, 5))
bars = plt.bar(
    labels,
    compute_values,
)

plt.ylabel("Median GPU compute-only (ms)")
plt.title(
    "GPU compute-only — Patch 7x7 / Search 21x21"
)

for bar, value in zip(
    bars,
    compute_values,
):
    plt.text(
        bar.get_x()
        + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.3f}",
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.show()

## 9. Benchmark steady-state end-to-end

End-to-end benchmark đo toàn bộ lời gọi implementation, có thể bao gồm:

- CPU padding;
- Host-to-Device transfer;
- GPU allocation;
- CUDA computation;
- Device-to-Host transfer;
- output conversion.

Warm-up được thực hiện trước để giảm ảnh hưởng của first-time CUDA initialization và kernel compilation.

In [ ]:
def run_v1_end_to_end():
    return nlm_gpu_v1(
        image=noisy_image,
        patch_size=config.patch_size,
        search_window_size=config.search_window_size,
        h=config.h,
        block_size=config.block_size,
    )


def run_v2_end_to_end():
    return nlm_gpu_v2(
        image=noisy_image,
        patch_size=config.patch_size,
        search_window_size=config.search_window_size,
        h=config.h,
        block_size=config.block_size,
    )


def run_v3_end_to_end():
    return nlm_gpu_v3(
        image=noisy_image,
        patch_size=config.patch_size,
        search_window_size=config.search_window_size,
        h=config.h,
        block_size=config.block_size,
        displacement_batch_size=V3_BATCH_SIZE,
    )

In [ ]:
v1_end_to_end = benchmark_gpu_end_to_end(
    function=run_v1_end_to_end,
    warmup_runs=5,
    measured_runs=20,
)

v2_end_to_end = benchmark_gpu_end_to_end(
    function=run_v2_end_to_end,
    warmup_runs=5,
    measured_runs=20,
)

v3_end_to_end = benchmark_gpu_end_to_end(
    function=run_v3_end_to_end,
    warmup_runs=5,
    measured_runs=20,
)

print("GPU V1 end-to-end:")
display(v1_end_to_end)

print("GPU V2 end-to-end:")
display(v2_end_to_end)

print("GPU V3 end-to-end:")
display(v3_end_to_end)

In [ ]:
v1_e2e_ms = (
    v1_end_to_end["median_seconds"]
    * 1000.0
)

v2_e2e_ms = (
    v2_end_to_end["median_seconds"]
    * 1000.0
)

v3_e2e_ms = (
    v3_end_to_end["median_seconds"]
    * 1000.0
)

end_to_end_table = pd.DataFrame(
    [
        {
            "Implementation": "GPU V1",
            "Median end-to-end (ms)": (
                v1_e2e_ms
            ),
            "Speedup so với V1": 1.0,
        },
        {
            "Implementation": "GPU V2",
            "Median end-to-end (ms)": (
                v2_e2e_ms
            ),
            "Speedup so với V1": (
                v1_e2e_ms / v2_e2e_ms
            ),
        },
        {
            "Implementation": "GPU V3",
            "Median end-to-end (ms)": (
                v3_e2e_ms
            ),
            "Speedup so với V1": (
                v1_e2e_ms / v3_e2e_ms
            ),
        },
    ]
)

end_to_end_table

In [ ]:
labels = ["GPU V1", "GPU V2", "GPU V3"]
e2e_values = [
    v1_e2e_ms,
    v2_e2e_ms,
    v3_e2e_ms,
]

plt.figure(figsize=(8, 5))
bars = plt.bar(
    labels,
    e2e_values,
)

plt.ylabel("Median end-to-end runtime (ms)")
plt.title(
    "Steady-state end-to-end — Patch 7x7 / Search 21x21"
)

for bar, value in zip(
    bars,
    e2e_values,
):
    plt.text(
        bar.get_x()
        + bar.get_width() / 2,
        bar.get_height(),
        f"{value:.3f}",
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.show()

## 10. Thí nghiệm batch size của GPU V3

`displacement_batch_size` là tham số thiết kế quan trọng của V3.

- Batch nhỏ → nhiều batch hơn → nhiều kernel launches hơn.
- Batch lớn → ít kernel launches hơn nhưng intermediate buffers lớn hơn và memory traffic có thể tăng.

Ta thử:

```text
8, 16, 32, 64
```

và đo **GPU compute-only** cho từng cấu hình.

In [ ]:
BATCH_SIZES = [8, 16, 32, 64]

batch_results = []

for batch_size in BATCH_SIZES:
    prepared = prepare_gpu_v3_pipeline_launch(
        image=noisy_image,
        patch_size=config.patch_size,
        search_window_size=config.search_window_size,
        h=config.h,
        block_size=config.block_size,
        displacement_batch_size=batch_size,
    )

    launcher = prepared[
        "kernel_launcher"
    ]

    # Pre-warm riêng cho cấu hình batch hiện tại.
    for _ in range(5):
        launcher()

    cp.cuda.get_current_stream().synchronize()

    result = benchmark_cuda_kernel(
        kernel_launcher=launcher,
        warmup_runs=5,
        measured_runs=20,
    )

    batch_results.append(
        {
            "Batch size": batch_size,
            "Số displacement": (
                prepared[
                    "number_of_displacements"
                ]
            ),
            "Số batch": (
                prepared["number_of_batches"]
            ),
            "Median compute-only (ms)": (
                result["median_ms"]
            ),
            "Mean (ms)": (
                result["mean_ms"]
            ),
            "Std (ms)": (
                result["std_ms"]
            ),
        }
    )

batch_table = pd.DataFrame(
    batch_results
)

batch_table

In [ ]:
best_batch_row = batch_table.loc[
    batch_table[
        "Median compute-only (ms)"
    ].idxmin()
]

best_batch_size = int(
    best_batch_row["Batch size"]
)

best_batch_runtime_ms = float(
    best_batch_row[
        "Median compute-only (ms)"
    ]
)

print(
    "Batch size tốt nhất trong sweep:",
    best_batch_size,
)

print(
    "Median compute-only tương ứng:",
    f"{best_batch_runtime_ms:.3f} ms",
)

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(
    batch_table["Batch size"],
    batch_table[
        "Median compute-only (ms)"
    ],
    marker="o",
)

plt.xlabel("Displacement batch size")
plt.ylabel("Median GPU compute-only (ms)")
plt.title("Ảnh hưởng của batch size đến GPU V3")
plt.xticks(BATCH_SIZES)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Benchmark lại V3 với batch size tốt nhất

Cell này giúp phân biệt:

- **V3 mặc định (`batch=32`)**
- **V3 sau khi tuning batch size**

Việc tuning chỉ thay đổi cách chia batch, không thay đổi thuật toán hay output.

In [ ]:
best_v3_prepared = prepare_gpu_v3_pipeline_launch(
    image=noisy_image,
    patch_size=config.patch_size,
    search_window_size=config.search_window_size,
    h=config.h,
    block_size=config.block_size,
    displacement_batch_size=best_batch_size,
)

best_v3_launcher = (
    best_v3_prepared["kernel_launcher"]
)

for _ in range(10):
    best_v3_launcher()

cp.cuda.get_current_stream().synchronize()

best_v3_compute_benchmark = benchmark_cuda_kernel(
    kernel_launcher=best_v3_launcher,
    warmup_runs=10,
    measured_runs=50,
)

best_v3_compute_ms = (
    best_v3_compute_benchmark[
        "median_ms"
    ]
)

print(
    "V3 batch mặc định:",
    V3_BATCH_SIZE,
    "→",
    f"{v3_compute_ms:.3f} ms",
)

print(
    "V3 batch tốt nhất:",
    best_batch_size,
    "→",
    f"{best_v3_compute_ms:.3f} ms",
)

print(
    "Speedup V3 tuned so với V1:",
    f"{v1_compute_ms / best_v3_compute_ms:.2f}×",
)

print(
    "Speedup V3 tuned so với V2:",
    f"{v2_compute_ms / best_v3_compute_ms:.2f}×",
)

## 11. Tổng hợp kết quả

In [ ]:
summary_table = pd.DataFrame(
    [
        {
            "Phiên bản": "GPU V1",
            "Ý tưởng chính": (
                "Song song hóa theo output pixel"
            ),
            "Compute-only (ms)": (
                v1_compute_ms
            ),
            "End-to-end (ms)": (
                v1_e2e_ms
            ),
        },
        {
            "Phiên bản": "GPU V2",
            "Ý tưởng chính": (
                "Shared Memory tile + halo"
            ),
            "Compute-only (ms)": (
                v2_compute_ms
            ),
            "End-to-end (ms)": (
                v2_e2e_ms
            ),
        },
        {
            "Phiên bản": (
                f"GPU V3 (batch={V3_BATCH_SIZE})"
            ),
            "Ý tưởng chính": (
                "Cross-correlation + batching"
            ),
            "Compute-only (ms)": (
                v3_compute_ms
            ),
            "End-to-end (ms)": (
                v3_e2e_ms
            ),
        },
        {
            "Phiên bản": (
                f"GPU V3 tuned "
                f"(batch={best_batch_size})"
            ),
            "Ý tưởng chính": (
                "Cross-correlation + tuned batching"
            ),
            "Compute-only (ms)": (
                best_v3_compute_ms
            ),
            "End-to-end (ms)": np.nan,
        },
    ]
)

summary_table

## 12. Hướng diễn giải kết quả

Không nên viết kết luận V3 trước khi xem runtime thực tế.

### Trường hợp A — V3 nhanh hơn V2

Có thể kết luận rằng reformulation bằng patch energy + cross-correlation, kết hợp batching, đã giảm chi phí repeated patch-distance computation đủ để bù cho intermediate memory traffic và multi-kernel overhead.

### Trường hợp B — V3 đúng nhưng vẫn chậm hơn V2

Đây vẫn là kết quả hợp lệ. Khi đó cần xem xét các bottleneck:

- tạo product maps;
- đọc/ghi intermediate buffers ở Global Memory;
- horizontal/vertical accumulation;
- số kernel launches còn lại;
- batch size;
- direct cross-correlation chưa đủ hiệu quả.

Kết quả này có thể dẫn tới bước tối ưu tiếp theo như kernel fusion hoặc FFT-based correlation.

### Trường hợp C — compute-only tốt nhưng end-to-end không tốt

Khi đó optimization ở CUDA computation có hiệu quả, nhưng:

- allocation;
- padding;
- H2D/D2H;
- preparation của intermediate buffers

đang chi phối tổng runtime.

### Điểm phải báo cáo

Notebook cuối cùng nên nêu đồng thời:

1. Correctness so với V1/V2.
2. PSNR/SSIM.
3. GPU compute-only runtime.
4. Steady-state end-to-end runtime.
5. Batch-size trade-off.
6. Speedup hoặc slowdown của V3 so với V1/V2.

## 13. Khung kết luận sau khi chạy notebook

Sau khi có số liệu, điền kết luận theo mẫu:

> GPU V3 reformulates NLM patch distance bằng patch energy và cross-correlation, đồng thời batch nhiều search displacement để giảm số nhóm CUDA kernel launch. Kết quả correctness cho thấy V3 ______ với V1/V2 trong tolerance đã chọn. Với cấu hình `512×512`, patch `7×7`, search `21×21`, GPU V3 đạt median compute-only ______ ms và end-to-end ______ ms. Batch size tốt nhất trong các giá trị thử nghiệm là ______. So với GPU V2, V3 ______. Kết quả cho thấy ______ là bottleneck/lợi ích chính của thiết kế cross-correlation hiện tại.

Không điền các chỗ trống trước khi chạy benchmark thực tế.